# Семинар 6. Матричное и автоматическое дифференцирование

Практическая часть по лекции №7 курса «Методы оптимизации в машинном обучении».

**Формат:** 5 основных задач примерно на 40 минут, ещё 5 дополнительных — для самостоятельного решения. Каждой программной задаче соответствует одноимённая задача в теоретическом комплекте.

В начале каждой задачи есть заготовки данных и функций, но **нет готовых решений**. Допишите помеченные участки `# TODO`, получите численные результаты и кратко объясните их в текстовых ячейках.

Примечание: `torch` — библиотека, позволяющая автоматически вычислять градиенты. На первых задачах сравниваем её результаты с вычислениями средствами `numpy`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

np.set_printoptions(precision=5, suppress=True)
torch.set_default_dtype(torch.float64)
rng = np.random.default_rng(42)

print('NumPy:', np.__version__, '| PyTorch:', torch.__version__)

## Основные задачи

### Задача 1. Градиент составной матричной функции

Рассмотрим масштабированную версию задачи на бумаге с $A\in\mathbb R^{300\times40}$ и положительными весами $c_i$:

$$F(w)=\frac{1}{2n}\sum_{i=1}^n c_i^2\big((Aw-b)_i\big)^2+\frac{\lambda}{2}\|w\|_2^2.$$

**а)** Реализуйте аналитически вычисленный градиент в `grad_1` без покоординатных циклов.

**б)** Получите градиент той же функции средствами автоматического дифференцирования PyTorch.

**в)** Сравните нормы градиентов и норму их разности. Какая точность согласования получилась?

**г)** Объясните, какие матричные размеры должны иметь все промежуточные объекты.

In [ ]:
n, d = 300, 40
A1 = rng.normal(size=(n, d)) / np.sqrt(d)
w_true_1 = rng.normal(size=d)
b1 = A1 @ w_true_1 + 0.1 * rng.normal(size=n)
c1 = rng.uniform(0.5, 1.5, size=n)
lam1 = 0.2
w1 = rng.normal(size=d)

def objective_1(w):
    r = A1 @ w - b1
    return 0.5 * np.mean((c1 * r)**2) + 0.5 * lam1 * np.dot(w, w)

def grad_1(w):
    # TODO: запишите матричный градиент
    return None

# TODO: реализуйте вычисление градиента через PyTorch и сравните с grad_1(w1)
# Проверка: форма аналитического градиента должна быть (d,).
print('F(w1) =', objective_1(w1))

### Задача 2. Разветвлённый граф: один вход влияет на несколько слагаемых

Теперь $x\in\mathbb R^{100}$, а индексы циклические: $x_{101}=x_1$.

$$F(x)=\sum_{i=1}^{100}(x_ix_{i+1}+x_i)(x_ix_{i+1})^2.$$

**а)** Вычислите градиент через PyTorch и проверьте результат конечной разностью по трём выбранным координатам.

**б)** Объясните, почему изменение $x_i$ отражается как минимум на двух соседних слагаемых суммы.

**в)** Для уменьшенной версии $d=4$ самостоятельно реализуйте обратный проход через введённые на бумаге промежуточные переменные и сравните результат с автоматическим.

**г)** Изобразите зависимость $F(x+te_i)$ от $t$ для выбранного $i$ вблизи нуля и сопоставьте наклон с производной.

In [ ]:
d2 = 100
x2_np = rng.normal(size=d2) / np.sqrt(d2)

def function_2_numpy(x):
    nxt = np.roll(x, -1)
    a = x * nxt
    return np.sum((a + x) * a**2)

def function_2_torch(x):
    nxt = torch.roll(x, shifts=-1, dims=0)
    a = x * nxt
    return ((a + x) * a**2).sum()

x2_t = torch.tensor(x2_np, requires_grad=True)
# TODO: запустите обратный проход и получите градиент
# TODO: проверьте 3 координаты конечной разностью
# TODO: сделайте собственный обратный проход для d=4
print('F(x) =', function_2_numpy(x2_np))

### Задача 3. Прямой и обратный режимы без полной матрицы Якоби

Пусть $G:\mathbb R^{100}\to\mathbb R^{10}$ задана функцией ниже.

Нужно вычислить $J_G(x)v$ и $J_G(x)^\top u$ **без явного построения** матрицы $J_G$.

**а)** Используйте прямой и обратный режимы PyTorch (подсказка: `torch.func.jvp`, `torch.func.vjp`).

**б)** Проверьте тождество

$$u^\top\big(J_G(x)v\big)=v^\top\big(J_G(x)^\top u\big).$$

**в)** Проверьте его для нескольких случайных $u,v$, измерьте абсолютную ошибку.

**г)** Укажите размерности обоих произведений.

In [ ]:
d3, m3 = 100, 10
A3 = torch.tensor(rng.normal(size=(m3, d3)) / np.sqrt(d3))
B3 = torch.tensor(rng.normal(size=(m3, d3)) / np.sqrt(d3))
x3 = torch.tensor(rng.normal(size=d3))
u3 = torch.tensor(rng.normal(size=m3))
v3 = torch.tensor(rng.normal(size=d3))

def G3(x):
    return torch.tanh(A3 @ x) + 0.1 * (B3 @ x)**2

# TODO: вычислите Jv прямым режимом
# TODO: вычислите J^T u обратным режимом
# TODO: сравните два скалярных произведения
print('Размерность выхода G:', tuple(G3(x3).shape))

### Задача 4. Численная устойчивость конечной разности

Рассмотрим $f(x)=\sqrt{1+x}-1$ и производную в точке $x=0$:

$$D_h=\frac{\sqrt{1+h}-1}{h}.$$

**а)** Реализуйте вычисление $D_h$ напрямую и через выражение после рационализации числителя из бумажной задачи.

**б)** Для $h=10^{-1},\ldots,10^{-16}$ сравните обе оценки с точным значением $f'(0)$.

**в)** Постройте в логарифмических координатах графики абсолютной ошибки в зависимости от $h$. Почему возникают разные кривые?

**г)** Найдите шаг, на котором прямое вычисление становится особенно неточным.

In [ ]:
h4 = 10.0 ** (-np.arange(1, 17))
truth4 = 0.5

# TODO: вычислите две математически эквивалентные версии D_h
raw4 = None
stable4 = None

if raw4 is not None and stable4 is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.loglog(h4, np.abs(raw4 - truth4), 'o-', label='Исходная формула')
    ax.loglog(h4, np.abs(stable4 - truth4), 's-', label='Рационализованная формула')
    ax.set_xlabel('Шаг h')
    ax.set_ylabel('Абсолютная ошибка')
    ax.grid(True, which='both', alpha=0.2)
    ax.legend()
    plt.show()
else:
    print('Заполните raw4 и stable4, после чего выполните ячейку повторно.')

### Задача 5. Матричное преобразование и нелинейность

Пусть $B\in\mathbb R^{200\times50}$, $u\in\mathbb R^{200}$ и

$$F(w)=u^\top\varphi(Bw),\qquad \varphi(t)=\sin t+\frac{t^2}{10}.$$

**а)** Реализуйте аналитический градиент по $w$, пользуясь матричным правилом цепочки.

**б)** Сравните его с градиентом, полученным PyTorch.

**в)** Объясните, почему результатом обратного прохода от $B w$ к $w$ будет вектор размерности $50$.

**г)** Сравните приблизительное время повторных вычислений двух вариантов (результат может зависеть от размера задачи и вычислительного окружения).

In [ ]:
m5, d5 = 200, 50
B5 = rng.normal(size=(m5, d5)) / np.sqrt(d5)
u5 = rng.normal(size=m5)
w5 = rng.normal(size=d5)

def phi5(t):
    return np.sin(t) + t**2 / 10

def F5(w):
    return np.dot(u5, phi5(B5 @ w))

def grad5_analytic(w):
    # TODO: аналитический градиент (без циклов)
    return None

# TODO: создайте аналог F5 с помощью torch, вызовите backward()
# TODO: сравните результаты и время вычисления
print('F5(w5) =', F5(w5))

## Дополнительные задачи

### Задача 6. Градиент при линейной замене переменных

Рассмотрим $f(w)=\tfrac12 w^\top H w-b^\top w$ и замену $w=Dz$.

**а)** Реализуйте градиенты по $w$ и $z$ без использования автоматического дифференцирования.

**б)** Численно проверьте тождество

$$\nabla_z f(Dz)=D^\top\nabla_w f(w).$$

**в)** Запустите несколько итераций градиентного спуска в координатах $w$ и $z$ с одинаковым численным шагом. Объясните, почему траектории после обратной замены переменных могут различаться.

In [ ]:
d6 = 20
scales6 = np.geomspace(0.3, 3.0, d6)
D6 = np.diag(scales6)
H6 = np.diag(np.geomspace(1.0, 25.0, d6))
b6 = rng.normal(size=d6)
z6 = rng.normal(size=d6)

def F6_w(w):
    return 0.5 * w @ (H6 @ w) - b6 @ w

# TODO: аналитические градиенты по w и z
# TODO: численная проверка и сравнение траекторий GD

### Задача 7. Какой режим выгоднее?

Рассмотрите три отображения: $\mathbb R^2\to\mathbb R^{100}$, $\mathbb R^{100}\to\mathbb R$ и $\mathbb R^{100}\to\mathbb R^{100}$.

**а)** Для каждого случая вычислите полную матрицу Якоби двумя способами: `torch.func.jacfwd` и `torch.func.jacrev`.

**б)** Сравните результаты и времена работы на нескольких повторах. Учитывайте, что для коротких функций накладные расходы могут быть существенными.

**в)** Сопоставьте наблюдения с количеством входов и выходов, сформулируйте вывод.

In [ ]:
from time import perf_counter
shapes7 = [(2, 100), (100, 1), (100, 100)]

for d, m in shapes7:
    A = torch.tensor(rng.normal(size=(m, d)) / np.sqrt(d))
    x = torch.tensor(rng.normal(size=d))
    f = lambda t: torch.tanh(A @ t)
    # TODO: заполните измерение времени jacfwd и jacrev
    print(f'Случай R^{d} -> R^{m}: матрица Якоби размерности {m} x {d}')

### Задача 8. Недифференцируемая точка

Исследуйте $f(x)=\max(0,x)$ при $x=0$.

**а)** Вычислите производную, которую возвращает PyTorch в нуле.

**б)** Сравните численные левую, правую и симметричную разности для нескольких $h$.

**в)** Объясните, почему результат автоматического дифференцирования в точке излома — соглашение реализации, а не обычная производная.

In [ ]:
xs8 = torch.tensor(0.0, requires_grad=True)
fs8 = torch.relu(xs8)
# TODO: выполните backward() и изучите grad
hs8 = 10.0 ** (-np.arange(1, 9))
# TODO: реализуйте левую, правую и симметричную конечные разности

### Задача 9. Затухающие и взрывающиеся градиенты

Пусть $z_0=x$, $z_{k+1}=az_k$, $F(x)=z_n$.

**а)** Постройте график $|F'(x)|$ по $n=1,\ldots,100$ для $a\in\{0.8,1,1.2\}$.

**б)** Для нескольких значений $a,n$ найдите градиенты посредством PyTorch и сравните с аналитической формулой из бумажной задачи.

**в)** Дополнительно исследуйте цепочку с $z_{k+1}=\tanh(az_k)$: как меняется градиент при увеличении глубины?

In [ ]:
ns9 = np.arange(1, 101)
a_values9 = [0.8, 1.0, 1.2]

# TODO: численный и аналитический градиенты для линейной цепочки
# TODO: график по n (предпочтительно логарифмическая ось y)
# TODO (дополнительно): цепочка tanh

### Задача 10. Логистический эмпирический риск и численная устойчивость

Рассмотрим

$$R(w)=\frac1n\sum_{i=1}^n\log\bigl(1+e^{-y_iw^\top x_i}\bigr)+\frac\lambda2\|w\|_2^2,\qquad y_i\in\{-1,1\}.$$

**а)** Реализуйте функцию риска и её аналитический градиент для случайной выборки.

**б)** Проверьте градиент средствами PyTorch.

**в)** Исследуйте, что происходит с наивным вычислением $\log(1+e^t)$ при большом положительном $t$. Сравните с устойчивым способом `np.logaddexp(0,t)` или `torch.nn.functional.softplus`.

**г)** Объясните, почему различия в численных реализациях важны при длинных цепочках вычислений.

In [ ]:
n10, d10 = 700, 20
X10 = rng.normal(size=(n10, d10))
y10 = rng.choice([-1.0, 1.0], size=n10)
lam10 = 0.1
w10 = rng.normal(size=d10)

def risk10(w):
    margins = y10 * (X10 @ w)
    return np.mean(np.logaddexp(0.0, -margins)) + 0.5 * lam10 * np.dot(w, w)

# TODO: аналитический градиент + torch-проверка
# TODO: численная устойчивость наивной и устойчивой softplus для t=10,100,1000
print('R(w10) =', risk10(w10))

---

## Что нужно сдать

Для каждой основной задачи: работающий код, результаты сравнения производных или график, а также **2–3 предложения интерпретации**. Дополнительные задачи выполняются по желанию.

При сравнении градиентов указывайте, какую норму ошибки используете; при измерении времени — размерность и вычислительное окружение.